Библиотека Pandas предоставляет большое количество возможностей для преобразований данных, однако иногда необходимо совершать более сложные манипуляции над столбцами. Например, из столбцов, содержащих в себе некоторый текст, необходимо специальным образом извлечь определённые слова, даты или числа.

Мы можем написать некоторую функцию, которая принимает на вход один элемент столбца, каким-то образом его обрабатывает и возвращает результат, после чего применить эту функцию к каждому элементу в столбце с помощью специального метода apply(). В результате применения этой функции будет возвращён объект Series, элементы которого будут представлять результат работы этой функции.



Рассмотрим пример. В наших данных есть столбец с адресами объектов недвижимости. Проблема этого столбца в том, что в нём слишком большое количество уникальных значений: почти на каждый объект недвижимости в таблице приходится свой уникальный адрес. Убедимся в этом, вычислив количество уникальных значений в столбце с помощью метода nunique():

In [2]:
import pandas as pd

melb_df = pd.read_csv('data/melb_data_ps.csv', sep=',')
melb_df['Address'].nunique()

13378

Если мы прогнозируем цену объекта, то такое большое количество возможных категорий может плохо сказаться на модели, которую мы бы хотели в дальнейшем построить на наших данных. Говорят, что такой признак, скорее всего, не имеет статистической значимости, потому что не позволяет разделить данные на группы, которые можно сравнить по целевому признаку.

In [3]:
def get_street_type(address):
    # Создаём список географических пометок exclude_list.
    execlut_list = ['N','S','W','E']
    # Метод split() разбивает строку на слова по пробелу.
    # В результате получаем список слов в строке и заносим его в переменную address_list.
    address_list = address.split(' ')
    # Обрезаем список, оставляя в нём только последний элемент,
    # потенциальный подтип улицы, и заносим в переменную street_type.
    street_type = address_list[-1]
    # Делаем проверку на то, что полученный подтип является географической пометкой.
    # Для этого проверяем его на наличие в списке exclude_list.
    if street_type in execlut_list:
        street_type = address_list[-2]
    return street_type
#display(melb_df)

street_types = melb_df['Address'].apply(get_street_type)
display(street_types)

0        St
1        St
2        St
3        La
4        St
         ..
13575    Cr
13576    Dr
13577    St
13578    St
13579    St
Name: Address, Length: 13580, dtype: object

Обратите внимание, что функция пишется для одного элемента столбца, а метод apply() применяется к каждому его элементу. Используемая функция обязательно должна иметь возвращаемое значение.

In [4]:
display(street_types.value_counts())

Address
St           8012
Rd           2825
Ct            612
Dr            447
Av            321
Gr            311
Pde           211
Pl            169
Cr            152
Cl            100
La             67
Bvd            53
Tce            47
Wy             40
Avenue         40
Cct            25
Hwy            24
Parade         15
Boulevard      13
Sq             11
Crescent        9
Cir             7
Strand          7
Esplanade       6
Grove           5
Grn             4
Fairway         4
Mews            4
Gdns            4
Righi           3
Crossway        3
Esp             2
Ridge           2
Victoria        2
Crofts          2
Athol           1
Highway         1
Cove            1
Grange          1
Res             1
Terrace         1
Qy              1
Glade           1
Nook            1
Eyrie           1
Loop            1
Dell            1
East            1
Summit          1
Grand           1
Gra             1
Hts             1
Outlook         1
Woodland        1
Ave             1
Co

Из данного вывода можно увидеть, что есть группа наиболее популярных подтипов улиц, а дальше частота подтипов быстро падает.

В таком случае давайте применим очень распространённый метод уменьшения количества уникальных категорий — выделим n подтипов, которые встречаются чаще всего, а остальные обозначим как 'other' (другие).

Для этого к результату метода value_counts применим метод nlargest(), который возвращает n наибольших значений из Series. Зададим n=10, т. е. мы хотим отобрать десять наиболее популярных подтипов. Извлечём их названия с помощью атрибута index, а результат занесём в переменную popular_stypes:

In [9]:
popular_streets = street_types.value_counts().nlargest(10).index
melb_df['StreetType'] = street_types.apply(lambda x : x if x in popular_streets else 'other')
melb_df = melb_df.drop('Address', axis=1)
display(melb_df)

,index,Suburb,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount,Coordinates,StreetType
0,0,Abbotsford,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067,...,202.0,126.0,1970,Yarra,-37.79960,144.99840,Northern Metropolitan,4019,"-37.7996, 144.9984",St
1,1,Abbotsford,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067,...,156.0,79.0,1900,Yarra,-37.80790,144.99340,Northern Metropolitan,4019,"-37.8079, 144.9934",St
2,2,Abbotsford,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067,...,134.0,150.0,1900,Yarra,-37.80930,144.99440,Northern Metropolitan,4019,"-37.8093, 144.9944",St
3,3,Abbotsford,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067,...,94.0,126.0,1970,Yarra,-37.79690,144.99690,Northern Metropolitan,4019,"-37.7969, 144.9969",other
4,4,Abbotsford,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067,...,120.0,142.0,2014,Yarra,-37.80720,144.99410,Northern Metropolitan,4019,"-37.8072, 144.9941",St
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13575,13575,Wheelers Hill,4,h,1245000.0,S,Barry,26/08/2017,16.7,3150,...,652.0,126.0,1981,NaN,-37.90562,145.16761,South-Eastern Metropolitan,7392,"-37.90562, 145.16761",Cr
13576,13576,Williamstown,3,h,1031000.0,SP,Williams,26/08/2017,6.8,3016,...,333.0,133.0,1995,NaN,-37.85927,144.87904,Western Metropolitan,6380,"-37.85927, 144.87904",Dr
13577,13577,Williamstown,3,h,1170000.0,S,Raine,26/08/2017,6.8,3016,...,436.0,126.0,1997,NaN,-37.85274,144.88738,Western Metropolitan,6380,"-37.85274, 144.88738",St
13578,13578,Williamstown,4,h,2500000.0,PI,Sweeney,26/08/2017,6.8,3016,...,866.0,157.0,1920,NaN,-37.85908,144.89299,Western Metropolitan,6380,"-37.85908, 144.89299",St


Таким образом, с помощью написания собственных функций и их комбинирования с методом apply() из библиотеки Pandas мы смогли извлечь информацию из признака с адресом и заменить на признак подтипа улиц

Примечание. Внимательный читатель наверняка обратит внимание на то, что мы допустили небольшую ошибку!

Если присмотреться, то в списке подтипов улиц street_types можно заметить подтипы, которые именуются различным образом, но при этом обозначают одинаковые вещи. Например, подтипы Av и Avenue, Bvd и Boulevard, Pde и Parade. Мы упустили данный момент, хотя в реальных задачах стоит обращать пристальное внимание на результаты преобразований и исправлять неточности в данных.

Такие ошибки в данных (обозначение идентичных категорий различными именами) являются одним из видов «грязных» данных.

Порой отследить такие неточности бывает очень сложно, а при наличии большого количества категорий (например, более ста) — практически невозможно.

Мы предлагаем вам самостоятельно разобраться с этой ошибкой: попробуйте написать функцию-преобразование (lambda-функцию-преобразование), которая возвращала бы вместо значений Avenue, Boulevard и Parade их топографические сокращения, и примените её к данным о подтипах улиц.

→ Обратите внимание, что данное преобразование необходимо применить до сокращения количества уникальных категорий.

In [31]:
display(melb_df['StreetType'])

def street_types_df(address):
   if address == 'Avenue':
      return 'Av'
   elif address == 'Boulevard':
      return 'Bvd'
   elif address == 'Parade':
      return 'Pde'
   else:
      return address

melb_df['StreetType'] = melb_df['StreetType'].apply(street_types_df)
display(melb_df['StreetType'].value_counts().index)


0           St
1           St
2           St
3        other
4           St
         ...  
13575       Cr
13576       Dr
13577       St
13578       St
13579       St
Name: StreetType, Length: 13580, dtype: object

Index(['St', 'Rd', 'Ct', 'Dr', 'other', 'Av', 'Gr', 'Pde', 'Pl', 'Cr', 'Cl'], dtype='object', name='StreetType')

Задание 4.2

Ранее, в задании 3.3, мы создали признак WeekdaySale в таблице melb_df — день недели продажи. Из полученных в задании результатов можно сделать вывод, что объекты недвижимости в Мельбурне продаются преимущественно по выходным (суббота и воскресенье).
Напишите функцию get_weekend(weekday), которая принимает на вход элемент столбца WeekdaySale и возвращает 1, если день является выходным, и 0 — в противном случае, и создайте столбец Weekend в таблице melb_df с помощью неё.

Примените эту функцию к столбцу и вычислите среднюю цену объекта недвижимости, проданного в выходные дни. Результат округлите до целых.

In [56]:
melb_df['Date'] = pd.to_datetime(melb_df['Date'], dayfirst=True)
melb_df['DayOfWeek'] = melb_df['Date'].dt.dayofweek

def get_weekend(weekday):
   weekend_days = [5,6]
   if weekday in weekend_days:
      return 1
   else:
      return 0

melb_df['DayOfWeek'] = melb_df['DayOfWeek'].apply(get_weekend)

price_of_weekend = melb_df[melb_df['DayOfWeek'] == 1]['Price']
display(round(price_of_weekend.mean()))

1081199

Задание 4.3

- Преобразуйте столбец SellerG с наименованиями риелторских компаний в таблице melb_df следующим образом: оставьте в столбце только 49 самых популярных компаний, а остальные обозначьте как 'other'.

- Найдите, во сколько раз минимальная цена объектов недвижимости, проданных компанией 'Nelson', больше минимальной цены объектов, проданных компаниями, обозначенными как 'other'. Ответ округлите до десятых.

In [67]:
display(melb_df.head())

def sort_top_sellers(sellers):
   top_sellers = melb_df['SellerG'].value_counts().nlargest(49).index
   if sellers in top_sellers:
      return sellers
   else: 
      return 'other'

melb_df['TopSellers'] = melb_df['SellerG'].apply(sort_top_sellers)
result = melb_df[melb_df['TopSellers'] == 'Nelson']['Price'].min() / melb_df[melb_df['TopSellers'] == 'other']['Price'].min()
print(round(result,1))

,index,Suburb,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount,Coordinates,StreetType,DayOfWeek,TopSellers
0,0,Abbotsford,2,h,1480000.0,S,Biggin,2016-12-03,2.5,3067,...,1970,Yarra,-37.7996,144.9984,Northern Metropolitan,4019,"-37.7996, 144.9984",St,1,Biggin
1,1,Abbotsford,2,h,1035000.0,S,Biggin,2016-02-04,2.5,3067,...,1900,Yarra,-37.8079,144.9934,Northern Metropolitan,4019,"-37.8079, 144.9934",St,0,Biggin
2,2,Abbotsford,3,h,1465000.0,SP,Biggin,2017-03-04,2.5,3067,...,1900,Yarra,-37.8093,144.9944,Northern Metropolitan,4019,"-37.8093, 144.9944",St,1,Biggin
3,3,Abbotsford,3,h,850000.0,PI,Biggin,2017-03-04,2.5,3067,...,1970,Yarra,-37.7969,144.9969,Northern Metropolitan,4019,"-37.7969, 144.9969",other,1,Biggin
4,4,Abbotsford,4,h,1600000.0,VB,Nelson,2016-06-04,2.5,3067,...,2014,Yarra,-37.8072,144.9941,Northern Metropolitan,4019,"-37.8072, 144.9941",St,1,Nelson


1.3


Представьте, что вы занимаетесь подготовкой данных о вакансиях с платформы hh.ru. В вашем распоряжении имеется таблица, в которой с помощью парсинга собраны резюме кандидатов. В этой таблице есть текстовый столбец «Опыт работы». Пример такого столбца представлен ниже в виде объекта Series. Структура текста в столбце фиксирована и не может измениться.

Напишите функцию get_experience(arg), аргументом которой является строка столбца с опытом работы. Функция должна возвращать опыт работы в месяцах. Не забудьте привести результат к целому числу.

Примечание. Обратите внимание, что опыт работы может выражаться только в годах или только в месяцах. Учтите это при построении своей функции.

При проверке мы будем применять вашу функцию к разным Series с помощью метода apply().

Пример результата работы функции get_experience:

In [69]:
import pandas as pd
test_series_1 = pd.Series([
    'Опыт работы 8 лет 3 месяца',
    'Опыт работы 3 года 5 месяцев',
    'Опыт работы 1 год 9 месяцев',
    'Опыт работы 3 месяца',
    'Опыт работы 6 лет'
])

test_series_2 = pd.Series([
    'Опыт работы 5 лет',
    'Опыт работы 5 месяцев',
    'Опыт работы 1 год 1 месяц',
    'Опыт работы 3 месяца',
    'Опыт работы 7 лет'
])


def get_experience(arg):
    arg = arg.replace('Опыт работы', '')
    parts = arg.split()
    
    year_word_list = ['год', 'года', 'лет']
    month_word_list = ['месяц','месяца','месяцев']
    
    total_month = 0
    for i in range(len(parts)):
        if parts[i] in year_word_list:
            total_month += int(parts[i-1])*12
        elif parts[i] in month_word_list:
            total_month += int(parts[i-1])
    return total_month 
 

print(test_series_1.apply(get_experience))

0    99
1    41
2    21
3     3
4    72
dtype: int64
